# 07 — Qwen2-VL (custom ViT + MLP + Qwen)

Model **3/4** on EcoExplain Condition C.

**Kernel (required):** `Python (ecoexplain .venv)` — project `.venv`, **not** Anaconda.

`combo_id`: `qwen_vl_custom_mlp_qwen`

Same 8 test patches as the other VLM notebooks. With `LAPTOP=True` this uses the small checkpoint; `PURGE_AFTER=True` deletes weights from disk after scoring.

Next: run `08_vlm_blip2.ipynb` after this cell finishes and the kernel is idle.


## 1. Kernel check and this combo


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.vlm_combos import records_for_split, resolve_checkpoint
from src.vlm_eval import collate, compare_reports, environment_report, eval_combo, get_combo

JSONL = ROOT / "dataset.jsonl"
SPLITS = ROOT / "data" / "splits"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

env = environment_report()
print("python      ", env.get("python"))
print("torch       ", env.get("torch"), env.get("device"))
print("transformers", env.get("transformers"))
print("Vision2Seq  ", env.get("has_vision2seq"), "ImageTextToText", env.get("has_image_text"))
if "anaconda" in (env.get("python") or "").lower() or not (env.get("has_vision2seq") or env.get("has_image_text")):
    raise RuntimeError(
        "Wrong Jupyter kernel. Kernel → Change Kernel → Python (ecoexplain .venv). Then Kernel → Restart."
    )

COMBO_ID = "qwen_vl_custom_mlp_qwen"
combo_meta = get_combo(COMBO_ID)
print("combo      ", combo_meta.combo_id)
print("family     ", combo_meta.family)
print("vision     ", combo_meta.vision_encoder)
print("projector  ", combo_meta.projector)
print("llm        ", combo_meta.text_decoder)
print("paper HF   ", combo_meta.hf_model_id)
print("laptop HF  ", combo_meta.laptop_hf_model_id)

if combo_meta.laptop_hf_model_id and "Qwen2-VL" in combo_meta.family and env.get("torchvision") is None:
    raise RuntimeError(
        "Qwen2-VL needs torchvision in the project venv. In a terminal run:\n"
        "  .venv/bin/pip install torchvision\n"
        "Then Kernel → Restart and run again."
    )


/Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


python       /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/.venv/bin/python
torch        2.8.0 mps
transformers 4.57.6
Vision2Seq   True ImageTextToText True
combo       qwen_vl_custom_mlp_qwen
family      Qwen2-VL
vision      Custom ViT (dynamic resolution)
projector   MLP / merger
llm         Qwen2
paper HF    Qwen/Qwen2-VL-7B-Instruct
laptop HF   Qwen/Qwen2-VL-2B-Instruct


## 2. Same evaluation batch (do not change ids if you want a fair 4-way table)


In [2]:
SPLIT = "test"
MAX_EVAL = 55
DRY_RUN = False
LAPTOP = True
PURGE_AFTER = True

records = records_for_split(JSONL, SPLITS, SPLIT)
batch = collate(records, ROOT, n=MAX_EVAL)
print("split", SPLIT, "n", len(batch), "ids", [x["id"] for x in batch])
print("classes", pd.Series([x["label"] for x in batch]).value_counts().to_dict())
print("--- prompt sample ---")
print(batch[0]["input_text"][:400])


split test n 55 ids ['p00052', 'p00086', 'p00575', 'p00701', 'p00850', 'p00863', 'p00883', 'p00937', 'p01162', 'p01206', 'p01415', 'p01621', 'p01648', 'p01988', 'p02128', 'p02139', 'p02334', 'p02668', 'p02674', 'p02803', 'p02890', 'p03067', 'p03122', 'p03228', 'p03243', 'p03515', 'p03538', 'p03592', 'p03635', 'p03689', 'p03801', 'p03867', 'p03991', 'p04040', 'p04059', 'p04145', 'p04174', 'p04203', 'p04268', 'p04399', 'p04508', 'p04577', 'p04671', 'p04777', 'p04806', 'p04892', 'p04898', 'p04996', 'p05015', 'p05082', 'p05117', 'p05122', 'p05384', 'p05511', 'p05771']
classes {'increase': 19, 'decline': 18, 'stable': 18}
--- prompt sample ---
Task: ground_forest_disturbance_in_spectral_evidence
Condition C evidence (measured on disk, analysis v2):
T0 (2022) terrain RGB (30×70 native, stretched for display): brightness=0.151, green-cover=1.00, uniform illumination; canopy-green dominates. T1 (2023) terrain RGB (30×70 native, stretched for display): brightness=0.169, green-cover=0.99, unifor

## 3. Run this VLM only


In [3]:
combo = get_combo(COMBO_ID)
resolved = resolve_checkpoint(combo, laptop=LAPTOP)
print("loading", resolved.hf_model_id)
report = eval_combo(
    combo,
    batch,
    RESULTS,
    dry_run=DRY_RUN,
    laptop=LAPTOP,
    purge_after=PURGE_AFTER,
)
from src.vlm_eval import format_accuracy_pct
print("wrote", RESULTS / f"vlm_combo_{COMBO_ID}.json")
print("accuracy", format_accuracy_pct(report.get("accuracy")))



loading Qwen/Qwen2-VL-2B-Instruct


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


loaded Qwen/Qwen2-VL-2B-Instruct as AutoModelForImageTextToText on mps
purged local cache /Users/vaibhav/.cache/huggingface/hub/models--Qwen--Qwen2-VL-2B-Instruct
purged local cache /Users/vaibhav/.cache/huggingface/hub/.locks/models--Qwen--Qwen2-VL-2B-Instruct
qwen_vl_custom_mlp_qwen: status=ok acc=56.4% n=55 parsed=55 s=817.73
wrote /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/results/vlm_combo_qwen_vl_custom_mlp_qwen.json
accuracy 56.4%


## 4. Scorecard


In [4]:
from src.vlm_eval import format_accuracy_pct

_report_json = RESULTS / f"vlm_combo_{COMBO_ID}.json"
if not _report_json.exists():
    raise FileNotFoundError(f"No results yet. Run the eval cell above first: {_report_json}")
report = json.loads(_report_json.read_text())

pd.DataFrame([{
    "combo_id": report["combo_id"],
    "hf_model_id": report.get("hf_model_id"),
    "status": report["status"],
    "accuracy (%)": format_accuracy_pct(report.get("accuracy")),
    "majority baseline (%)": format_accuracy_pct(report.get("majority_accuracy")),
    "n_eval": report.get("n_eval"),
    "n_parsed": report.get("n_parsed_class"),
    "seconds": report.get("seconds"),
    "error": (report.get("error") or "")[:200],
}])



,combo_id,hf_model_id,status,accuracy (%),majority baseline (%),n_eval,n_parsed,seconds,error
0,qwen_vl_custom_mlp_qwen,Qwen/Qwen2-VL-2B-Instruct,ok,56.4%,34.5%,55,55,817.73,


## 5. Predictions


In [5]:
if "report" not in globals():
    _report_json = RESULTS / f"vlm_combo_{COMBO_ID}.json"
    report = json.loads(_report_json.read_text())

pred_df = pd.DataFrame(report["predictions"])
display(pred_df[["id", "gold", "pred", "correct"]])
print("--- first pred_text ---")
print((pred_df["pred_text"].iloc[0] or "")[:800])


,id,gold,pred,correct
0,p00052,decline,decline,True
1,p00086,stable,stable,True
2,p00575,decline,decline,True
3,p00701,increase,stable,False
4,p00850,stable,stable,True
5,p00863,stable,stable,True
6,p00883,decline,decline,True
7,p00937,decline,decline,True
8,p01162,increase,stable,False
9,p01206,stable,decline,False


--- first pred_text ---
decline


When this kernel is **idle** (not `[*]`), open `08_vlm_blip2.ipynb` and run all cells. Do not start the next VLM while this one is still downloading.
